# map

> Generate a markmap-ready Markdown course map from cached summaries.json files

In [ ]:
#| default_exp map

In [ ]:
#| hide
from nbdev.showdoc import *

## Design

Read the new self-contained `summaries.json` for each video in an id-list file, flatten to per-section records, and emit a single Markdown document with three views:

- **By Lecture** — every lesson with its sections in playback order
- **By Topic** — keywords occurring in ≥N distinct lessons (cross-cutting themes)
- **By Keyword** — every normalized keyword with all sections that mention it

The output is a YAML-frontmatter Markdown ready for `markmap-cli` (interactive folding mindmap in browser) and also valid for `pandoc` (static linked HTML).

Only `summaries.json` is read — no `meta.json` or `toc.json` access. The video URL and section start times come straight from the embedded `video.url` and `sections[].start`.

In [ ]:
#| export
import json
import re
from collections import Counter, defaultdict
from pathlib import Path
from pydantic import Field
from yttoc.summarize import AssembledSection, AssembledSummaries


In [ ]:
#| export
class FlattenedSection(AssembledSection):
    "One row of the cross-video keyword/topic grid. Inherits AssembledSection fields and adds video context."
    lesson: int = Field(ge=1, description="1-based index in the lesson list")
    video_id: str = Field(description="YouTube video ID")
    video_title: str = Field(description="Video title")
    jump_url: str = Field(description="Deep-link URL for the section start")

def _norm_kw(kw: str # Raw keyword from summaries.json
            ) -> str: # Lowercased, alphanumeric-only key for grouping
    "Normalize a keyword for grouping (lowercase + drop non-alphanumerics so 'FastHTML' and 'fast html' merge)."
    return re.sub(r'[^a-z0-9]+', '', kw.lower())

def _section_label(row: FlattenedSection # Flattened section record
                  ) -> str: # 'L{lesson} §{path} {title}'
    "Short human label for a section across the whole course."
    return f"L{row.lesson} §{row.path} {row.title}"

def _canonical(originals: list[str]) -> str:
    "Pick the most frequent original casing for a normalized key. Ties resolved lexicographically (deterministic)."
    counts = Counter(originals)
    # Iterating sorted unique strings makes max() pick the lexicographically smallest on ties.
    return max(sorted(set(originals)), key=counts.get)

def load_summaries(video_ids: list[str], # Ordered video ids (lesson 1, 2, ...)
                   root: Path # Cache root dir
                  ) -> list[tuple[int, AssembledSummaries]]: # Lesson-tagged summaries
    "Load each video's summaries.json and pair with lesson number from list order."
    docs = []
    for i, vid in enumerate(video_ids, 1):
        p = Path(root) / vid / 'summaries.json'
        if not p.exists(): continue
        doc = AssembledSummaries.model_validate_json(p.read_text(encoding='utf-8'))
        docs.append((i, doc))
    return docs

def flatten_sections(docs: list[tuple[int, AssembledSummaries]] # Lesson-tagged summaries
                    ) -> list[FlattenedSection]: # One FlattenedSection per section with video context
    "Flatten all docs into one section-level list with lesson/video metadata attached."
    rows = []
    for lesson, doc in docs:
        v = doc.video
        url = v.url or ''
        for sec in doc.sections:
            jump = f"{url}&t={sec.start}" if url else ''
            rows.append(FlattenedSection(
                **sec.model_dump(),
                lesson=lesson,
                video_id=v.id,
                video_title=v.title,
                jump_url=jump,
            ))
    return rows


In [ ]:
#| export
def render_by_lecture(docs: list[tuple[int, AssembledSummaries]] # Lesson-tagged summaries
                     ) -> str: # Markdown fragment
    "Render the By Lecture view (Lesson → sections in playback order)."
    lines = ['## By Lecture', '']
    for lesson, doc in docs:
        v = doc.video
        url = v.url or ''
        lines.append(f"- Lesson {lesson}: {v.title}")
        for sec in doc.sections:
            jump = f"{url}&t={sec.start}" if url else '#'
            lines.append(f"  - [{sec.path}. {sec.title}]({jump})")
    return '\n'.join(lines)

def _build_keyword_index(rows: list[FlattenedSection] # Flattened section rows
                        ) -> tuple[dict, dict]: # (norm → [(orig, row), ...], norm → set of lesson nums)
    "Group rows by normalized keyword, also tracking distinct lesson coverage."
    idx = defaultdict(list)
    by_lessons = defaultdict(set)
    for row in rows:
        for kw in row.keywords:
            n = _norm_kw(kw)
            if not n: continue
            idx[n].append((kw, row))
            by_lessons[n].add(row.lesson)
    return idx, by_lessons

def render_by_topic(rows: list[FlattenedSection], # Flattened section rows
                    min_lessons: int = 2 # Minimum distinct lessons for inclusion
                   ) -> str: # Markdown fragment
    "Render the By Topic view: keywords occurring in ≥ min_lessons distinct lessons."
    idx, by_lessons = _build_keyword_index(rows)
    cross = {n: pairs for n, pairs in idx.items() if len(by_lessons[n]) >= min_lessons}
    lines = ['## By Topic',
            f'_Keywords spanning ≥{min_lessons} lessons_', '']
    if not cross:
        lines.append('_(no cross-lesson topics yet)_')
        return '\n'.join(lines)
    # Sort: most lessons first, then most occurrences, then alphabetical
    sorted_keys = sorted(cross,
                         key=lambda n: (-len(by_lessons[n]), -len(cross[n]), n))
    for n in sorted_keys:
        display = _canonical([orig for orig, _ in cross[n]])
        lines.append(f"- {display} _(in {len(by_lessons[n])} lessons)_")
        for _orig, row in cross[n]:
            lines.append(f"  - [{_section_label(row)}]({row.jump_url or '#'})")
    return '\n'.join(lines)

def render_by_keyword(rows: list[FlattenedSection] # Flattened section rows
                     ) -> str: # Markdown fragment
    "Render the By Keyword view: every normalized keyword with its sections."
    idx, _ = _build_keyword_index(rows)
    lines = ['## By Keyword', '']
    for n in sorted(idx):
        display = _canonical([orig for orig, _ in idx[n]])
        lines.append(f"- {display}")
        for _orig, row in idx[n]:
            lines.append(f"  - [{_section_label(row)}]({row.jump_url or '#'})")
    return '\n'.join(lines)

_FRONTMATTER = """---
markmap:
  initialExpandLevel: 2
  maxWidth: 320
---"""

def render_map(docs: list[tuple[int, AssembledSummaries]], # Lesson-tagged summaries
               title: str = 'Course Learning Map', # Top-level heading
               min_topic_lessons: int = 2 # Threshold for By Topic inclusion
              ) -> str: # Full Markdown document
    "Render the full course map Markdown (frontmatter + 3 views)."
    rows = flatten_sections(docs)
    parts = [
        _FRONTMATTER, '',
        f'# {title}', '',
        render_by_lecture(docs), '',
        render_by_topic(rows, min_lessons=min_topic_lessons), '',
        render_by_keyword(rows),
    ]
    return '\n'.join(parts)


## CLI

In [ ]:
#| export
from fastcore.script import call_parse, Param
from yttoc.fetch import _DEFAULT_ROOT

@call_parse
def yttoc_map(ids: Param('Cached video IDs in lesson order (1+ required)', str, nargs='+'),
              title: str = 'Course Learning Map', # Top-level heading
              root: str = None, # Root cache directory
              min_topic_lessons: int = 2, # By Topic threshold (distinct lessons)
             ):
    "Generate a markmap-ready Markdown course map from cached summaries.json files."
    root = Path(root) if root else _DEFAULT_ROOT
    docs = load_summaries(ids, root)
    if not docs:
        raise SystemExit(f"No summaries.json found for: {ids}")
    print(render_map(docs, title=title, min_topic_lessons=min_topic_lessons))

## Tests

In [ ]:
# Test 1: _norm_kw lowercases and strips ALL non-alphanumerics so variants merge
assert _norm_kw('FastHTML') == 'fasthtml'
assert _norm_kw('fast HTML') == 'fasthtml'   # whitespace dropped
assert _norm_kw('Tool-loop!') == 'toolloop'  # punctuation dropped
assert _norm_kw('  ') == ''
print('ok')

In [ ]:
# Test 1b: _canonical is deterministic on ties (lexicographic) regardless of input order
assert _canonical(['FastHTML', 'fast html']) == _canonical(['fast html', 'FastHTML'])
assert _canonical(['FastHTML', 'fast html']) == 'FastHTML'  # 'F' < 'f' in ASCII
# Higher-frequency wins regardless of order
assert _canonical(['foo', 'foo', 'Foo']) == 'foo'
assert _canonical(['Foo', 'foo', 'foo']) == 'foo'
print('ok')

In [ ]:
# Test 3: load_summaries skips missing, attaches lesson number from order
from tempfile import TemporaryDirectory

def _make_doc(vid, title, sections):
    from yttoc.summarize import AssembledSummaries, VideoBlock, AssembledSection
    return AssembledSummaries(
        video=VideoBlock(id=vid, title=title, channel='C',
                         url=f'https://youtu.be/{vid}', duration=600, upload_date='20260101'),
        sections=[AssembledSection(**s) if isinstance(s, dict) else s for s in sections],
        full={'summary': f'{title} overall.', 'keywords': ['overview'],
              'evidence': {'text': 'x', 'at': 0}},
    )

with TemporaryDirectory() as d:
    root = Path(d)
    (root / 'A').mkdir()
    (root / 'A' / 'summaries.json').write_text(_make_doc('A', 'Lesson A', [
        {'path': '1', 'title': 'IntroA', 'start': 0, 'end': 100,
         'summary': 's1', 'keywords': ['Git', 'FastHTML'], 'evidence': {'text': 't', 'at': 0}}]).model_dump_json())
    # B is intentionally missing summaries.json
    (root / 'B').mkdir()
    (root / 'C').mkdir()
    (root / 'C' / 'summaries.json').write_text(_make_doc('C', 'Lesson C', [
        {'path': '1', 'title': 'IntroC', 'start': 0, 'end': 200,
         'summary': 's2', 'keywords': ['fast html'], 'evidence': {'text': 't', 'at': 0}}]).model_dump_json())

    docs = load_summaries(['A', 'B', 'C'], root)
    assert len(docs) == 2
    assert docs[0][0] == 1  # (lesson, AssembledSummaries)
    assert docs[1][0] == 3  # lesson number reflects original list position
print('ok')


In [ ]:
# Test 4: flatten_sections attaches video context and builds jump_url
doc = _make_doc('VID', 'Lesson X', [
    {'path': '1', 'title': 'A', 'start': 0, 'end': 100,
     'summary': '', 'keywords': [], 'evidence': {'text': '', 'at': 0}},
    {'path': '2', 'title': 'B', 'start': 137, 'end': 300,
     'summary': '', 'keywords': [], 'evidence': {'text': '', 'at': 0}},
])
rows = flatten_sections([(1, doc)])
assert len(rows) == 2
assert rows[0].jump_url == 'https://youtu.be/VID&t=0'
assert rows[1].jump_url == 'https://youtu.be/VID&t=137'
assert rows[0].lesson == 1
assert rows[0].video_title == 'Lesson X'
print('ok')


In [ ]:
# Test 5: render_by_lecture lists sections under each lesson with deep-link URLs
docs = [
    (1, _make_doc('A', 'Lesson A', [
        {'path': '1', 'title': 'IntroA', 'start': 0, 'end': 100,
         'summary': '', 'keywords': [], 'evidence': {'text': '', 'at': 0}}])),
    (2, _make_doc('B', 'Lesson B', [
        {'path': '1', 'title': 'IntroB', 'start': 0, 'end': 100,
         'summary': '', 'keywords': [], 'evidence': {'text': '', 'at': 0}},
        {'path': '2', 'title': 'MainB', 'start': 200, 'end': 400,
         'summary': '', 'keywords': [], 'evidence': {'text': '', 'at': 0}}])),
]
out = render_by_lecture(docs)
assert '## By Lecture' in out
assert '- Lesson 1: Lesson A' in out
assert '- Lesson 2: Lesson B' in out
assert '[1. IntroA](https://youtu.be/A&t=0)' in out
assert '[2. MainB](https://youtu.be/B&t=200)' in out
print('ok')


In [ ]:
# Test 6: render_by_topic includes only keywords spanning >= min_lessons distinct lessons
from yttoc.map import FlattenedSection
rows = [
    FlattenedSection(path='1', title='A', start=0, end=10,
                     summary='', keywords=['Git', 'FastHTML', 'unique-to-1'],
                     evidence={'text': '', 'at': 0},
                     lesson=1, video_id='v1', video_title='V1', jump_url='u1'),
    FlattenedSection(path='2', title='B', start=0, end=10,
                     summary='', keywords=['Git'],
                     evidence={'text': '', 'at': 0},
                     lesson=1, video_id='v1', video_title='V1', jump_url='u2'),
    FlattenedSection(path='1', title='C', start=0, end=10,
                     summary='', keywords=['Git', 'fast html'],
                     evidence={'text': '', 'at': 0},
                     lesson=2, video_id='v2', video_title='V2', jump_url='u3'),
]
out = render_by_topic(rows, min_lessons=2)
assert '## By Topic' in out
assert 'Git' in out  # 2 lessons
assert 'FastHTML' in out or 'fast html' in out  # spans L1 and L2 after norm
assert 'unique-to-1' not in out  # only 1 lesson
print('ok')


In [ ]:
# Test 7: render_by_keyword groups every normalized keyword across all lessons
from yttoc.map import FlattenedSection
rows = [
    FlattenedSection(path='1', title='A', start=0, end=10,
                     summary='', keywords=['Solveit'],
                     evidence={'text': '', 'at': 0},
                     lesson=1, video_id='v1', video_title='V1', jump_url='u1'),
    FlattenedSection(path='3', title='B', start=0, end=10,
                     summary='', keywords=['solveit', 'agent'],
                     evidence={'text': '', 'at': 0},
                     lesson=2, video_id='v2', video_title='V2', jump_url='u2'),
]
out = render_by_keyword(rows)
lines = out.splitlines()
# Both Solveit-cased rows should land under the same group
solveit_idx = next(i for i, l in enumerate(lines) if l.startswith('- ') and 'olveit' in l.lower())
# Two children expected
assert lines[solveit_idx + 1].startswith('  - [L1 §1 A]')
assert lines[solveit_idx + 2].startswith('  - [L2 §3 B]')
print('ok')


In [ ]:
# Test 8: render_map composes frontmatter + title + 3 sections
docs = [
    (1, _make_doc('A', 'Lesson A', [
        {'path': '1', 'title': 'IntroA', 'start': 0, 'end': 100,
         'summary': '', 'keywords': ['Git'], 'evidence': {'text': '', 'at': 0}}])),
    (2, _make_doc('B', 'Lesson B', [
        {'path': '1', 'title': 'IntroB', 'start': 0, 'end': 100,
         'summary': '', 'keywords': ['Git'], 'evidence': {'text': '', 'at': 0}}])),
]
md = render_map(docs, title='Test Map', min_topic_lessons=2)
assert md.startswith('---\nmarkmap:')
assert '# Test Map' in md
assert '## By Lecture' in md
assert '## By Topic' in md
assert '## By Keyword' in md
assert 'Git' in md
print('ok')


In [ ]:
# Test 9: render_map directly (no stdout capture needed)
from yttoc.map import render_map

doc_a = _make_doc('A', 'Lesson A', [
    {'path': '1', 'title': 'IntroA', 'start': 0, 'end': 100,
     'summary': '', 'keywords': ['Git'], 'evidence': {'text': '', 'at': 0}}])

out = render_map([(1, doc_a)])
assert '# Course Learning Map' in out
assert '- Lesson 1: Lesson A' in out
assert 'IntroA' in out
print('ok')


## Generating a course map

`yttoc-map` reads cached `summaries.json` files and emits Markdown to stdout. The video IDs are positional args, so any shell can construct the list — from a file, a directory listing, or a literal sequence.

```bash
# From an id-list file (one id per line)
yttoc-map $(cat course-ids.txt) --title "My Course Learning Map" > course-map.md

# From every cached video
yttoc-map $(ls ~/.cache/yttoc) > all-cached-map.md

# Inline / ad-hoc
yttoc-map <VIDEO_ID_1> <VIDEO_ID_2> > two-lessons.md
```

Convert the Markdown to an interactive folding mindmap (single offline HTML):

```bash
npx -y markmap-cli@latest --offline course-map.md -o course-map.html
xdg-open course-map.html   # or: open course-map.html
```

For a static linked HTML page (no JS folding), use pandoc instead:

```bash
pandoc course-map.md -s --toc -o course-map.html
```

Tune the By Topic view by raising the cross-lesson threshold:

```bash
yttoc-map $(cat course-ids.txt) --min-topic-lessons 3 > course-map.md
```

## Searching with fzf

`yttoc-map` is for **browsing** (full overview, folding mindmap). For **finding** — "where is FastHTML mentioned across the whole course?" — pipe the cached `summaries.json` files through `jq` into `fzf` instead. Markmap has browser-side search, but interactive incremental search across hundreds of sections is much faster in the terminal.

Drop this into `~/.bashrc`:

```bash
yttoc-find() {
    for id in "$@"; do
        jq -r --arg id "$id" '
            .video.url as $u
            | .sections[]
            | [
                ($u + "&t=" + (.start|tostring)),
                ($id + " §" + .path + " " + .title),
                .summary,
                (.keywords | join(", "))
            ] | @tsv
        ' "$HOME/.cache/yttoc/$id/summaries.json"
    done \
    | fzf --delimiter=$'\t' \
          --with-nth=2 \
          --preview 'printf "%s\n\nKeywords: %s\n" {3} {4}' \
          --preview-window=down:60%:wrap \
          --bind 'enter:execute(xdg-open {1})'
}
```

Usage — same shell-expansion style as `yttoc-map`:

```bash
yttoc-find $(cat course-ids.txt)
yttoc-find $(ls ~/.cache/yttoc)
yttoc-find <VIDEO_ID_1> <VIDEO_ID_2>
```

Fuzzy-search section labels in the left pane, see the summary and keywords in the preview pane below, hit Enter to open the deep-linked YouTube URL in your browser. Replace `xdg-open` with `open` on macOS.

The fzf snippet stays out of the Python package on purpose: `xdg-open`/`open` is environment-specific, fzf isn't a Python dep, and shell composition is what shells are good at. yttoc generates the data; your shell does the searching.

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()